# Automation incidents: failure patterns, costs, and governance gaps

**Claim cluster:** `C033`, `C034`, `C035`, `C036`, `C037`, `C038`, `C039`, `C040`, `C041`, `C042`, `C097`, `C098`  
**Source of truth:** `data/AutomationResearch.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "AutomationResearch.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/AutomationResearch.duckdb (read-only).
CLAIM_IDS = ('C033', 'C034', 'C035', 'C036', 'C037', 'C038', 'C039', 'C040', 'C041', 'C042', 'C097', 'C098')
assert viz.DEFAULT_DB_PATH.endswith("AutomationResearch.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

C033-C038 describe five publicly documented incidents (RPA bot failures, duplicate payments, corruption of customer profiles, deployment breaks, access-removal gaps). C039-C042 and C097-C098 generalize from them to operational and governance problems.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Incident claims mapped to documented failures

Five incidents (S25-S29) are referenced by C033-C038. The metrics M52-M56 record specific numbers (duplicate payment $480,000; 15,000 profiles; 55 of 56 decommissioned bots; 90-day corruption).

In [ ]:
# Incident mapping claim -> metric
INCIDENT_MAP = {
    "C033": ("M53", 40),  # 40 bots at insurer
    "C034": ("M53", None),
    "C035": ("M52", 480000),
    "C036": ("M54", 15000),
    "C037": ("M55", None),
    "C038": ("M56", 55),
}

df = pd.DataFrame([
    {"claim": c, "metric": m, "stated": s}
    for c, (m, s) in INCIDENT_MAP.items()
])
df["metric_name"] = df["metric"].map(lambda m: viz.metrics_by_id([m])["metric_name"].iloc[0] if m else "—")
df["confidence"] = df["claim"].map(lambda c: claims.set_index("local_id").loc[c, "confidence"])
display(HTML("<h4>Incident claim -> metric mapping</h4>"))
show(df.rename(columns={"claim":"claim","metric":"metric_id","stated":"stated_val","metric_name":"name","confidence":"conf"}))

## Supporting context

### Incident metrics

M52 ($480,000 duplicate payment, S26), M53 (40 RPA bots, S25), M54 (15,000 corrupted profiles, S27), M55 (90-day silent corruption, S27), M56 (55/56 decommissioned bots, S29), M58 (30-50% RPA failure rate, S25), M59 (70-75% maintenance budget share, S25), M60 (50% project failure, S25), M82 (800ms network timeout, S26), M83 ($480,000 duplicate, S26), M89 (bot failure ~11 PM, S25), M90 (discovered next morning, S25), M91 (90 days, S27).

In [ ]:
display(HTML("<h4>Incident-related metrics (selected)</h4>"))
_inc = viz.metrics_by_id(["M52","M53","M54","M55","M56","M58","M59","M60"])
show(_inc[["local_id","metric_name","value","unit","confidence"]].rename(
    columns={"local_id":"m","metric_name":"name","confidence":"conf"}))

# Rates share one unit (percent), so they chart on one axis. Stated ranges
# are spanned via bar base (30->50, 70->75), never averaged into a midpoint.
RATES = [
    ("Human review rate (M36)", 0, 20, "20%", "medium"),
    ("Retry rate (M37)", 0, 15, "15%", "medium"),
    ("RPA project failure rate (M58)", 30, 20, "30-50%", "medium"),
    ("RPA maintenance budget share (M59)", 70, 5, "70-75%", "medium"),
]
fig = go.Figure(go.Bar(
    y=[r[0] for r in RATES],
    base=[r[1] for r in RATES],
    x=[r[2] for r in RATES],
    orientation="h",
    text=[r[3] for r in RATES],
    textposition="outside",
    marker_color=[viz.CONFIDENCE_COLORS[r[4]] for r in RATES],
))
fig.update_layout(
    title="Operational failure and labor rates (%, as stated; ranges spanned, not averaged)",
    xaxis_title="percent",
    height=320)
fig.show()

## Uncertainty and gaps

### Incident claim limitations

- **C033 (40 bots at insurer) and C034 (4 bots broken by UI update) both reference S25** but describe different failure modes (operational scale vs. vendor-change fragility).
- **C035 ($480,000 duplicate payment) is a `documented fact` (high)** from S26; C026 (Temporal $50/million) and M26 ($133/month hosted cost) are from a different prompt. They are not directly comparable.
- **C036 (15,000 profiles, 90 days) and M91 (90 days) refer to the same incident** but M55 gives the duration; M54 the profile count.

In [ ]:
# Incident claims that describe patterns rather than specific metrics
_patterns = ["C039","C040","C041","C042","C097","C098"]
display(HTML("<h4>Generalized incident / governance claims</h4>"))
show(claims.loc[claims["local_id"].isin(_patterns),
                ["local_id","claim_type","confidence","claim_text"]].rename(
    columns={"local_id":"claim","claim_type":"type","confidence":"conf"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Incident-to-metric links are authored from source references (S25-S29); claim text from `viz.fetch_claims`; metrics from `viz.metrics_by_id`. The mapping table prints both the claim id and the metric id for audit. The rate figure charts M36, M37, M58, M59 as stated (ranges spanned via bar base, not averaged).

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()